# 第四阶段：STL（标准模板库）

## 实验 6：`std::optional` —— 显式表达“可能没有值”

一些查询正常情况下就可能找不到结果。若用 `-1`、空字符串等 magic value 表示缺失，调用方必须记住隐藏约定；若返回裸指针，又会引入 null、ownership 和 lifetime 问题。

`std::optional<T>` 把“存在一个 `T`”或“不存在值”写进返回类型。本实验关注：

- optional 的 engaged/disengaged 状态如何建立和切换；
- 如何安全读取值，以及 `value_or()` 实际返回什么；
- optional 是否拥有其中的对象，引用何时失效；
- “没有结果”和“执行失败”为什么不是同一种语义；
- 如何把 C++ optional 映射到 C ABI 与 Kotlin nullable 类型。

In [ ]:
// 本步骤：引入 optional、字符串、断言和 C ABI 示例所需的标准库。
#include <cassert>
#include <cstddef>
#include <iostream>
#include <optional>
#include <stdexcept>
#include <string>
#include <string_view>

### 1. 返回类型直接表达可能缺失

`std::optional<int>` 有两个合法状态：

```text
engaged                         disengaged
┌──────────────────┐           ┌──────────────────┐
│ contains an int  │           │ contains no int  │
└──────────────────┘           └──────────────────┘
```

它没有规定缺失必须对应哪个整数，因此所有 `int` 值仍可作为正常结果。optional 本身不说明“为什么缺失”；适合查找不到、配置未提供等预期状态。

In [ ]:
// 本步骤：定义返回 optional 的查询函数，用 nullopt 表示正常的“未找到”。
std::optional<int> find_age(std::string_view name)
{
    // 命中已知名称时，返回值会直接构造 engaged optional。
    if (name == "Bob")
    {
        return 20;
    }

    if (name == "Alice")
    {
        return 21;
    }

    // 未命中不是执行错误，因此返回显式的空状态。
    return std::nullopt;
}

In [ ]:
// 本步骤：分别处理存在和缺失结果，不依赖 magic value。
{
    const std::optional<int> bob = find_age("Bob");

    // bool 检查与 has_value() 都是在查询是否包含值。
    if (bob.has_value())
    {
        std::cout << "Bob age = " << bob.value() << '\n';
    }

    const std::optional<int> unknown = find_age("Unknown");

    // 空 optional 的布尔转换为 false，不会访问不存在的 int。
    if (!unknown)
    {
        std::cout << "User not found\n";
    }

    assert(bob.has_value());
    assert(!unknown.has_value());
}

### 2. 建立和切换状态

默认构造或使用 `std::nullopt` 会得到 disengaged optional。赋值或 `emplace()` 会建立 contained value；`reset()` 会销毁 contained value 并恢复为空。

`emplace(args...)` 直接在 optional 内部构造 `T`。optional 不需要另行分配一个 `T`，但 `T` 自己仍可能管理动态资源，例如 `std::string` 的字符存储。

In [ ]:
// 本步骤：依次观察空、构造值、清空和重新赋值四种状态。
{
    std::optional<std::string> language;

    // 默认构造不包含 string。
    assert(!language);

    // 在 optional 内直接构造 string，并通过箭头访问其成员。
    language.emplace("Kotlin");
    assert(language->size() == 6);

    // reset 立即销毁 contained string。
    language.reset();
    assert(!language.has_value());

    // 普通赋值重新进入 engaged 状态。
    language = "C++";
    assert(language.value() == "C++");
}

### 3. 先检查，再访问

常见访问方式：

- `if (result)` / `has_value()`：只检查是否存在；
- `*result` / `result->member`：要求值已经存在，不额外检查；
- `value()`：空状态时抛出 `std::bad_optional_access`；
- `value_or(fallback)`：返回 contained value 或 fallback 的一个值副本。

不能通过解引用空 optional 来“测试”行为。那违反接口前置条件，不是安全的失败案例。

In [ ]:
// 本步骤：对比已检查访问、默认值访问和抛异常访问。
{
    const std::optional<int> present = 42;
    const std::optional<int> missing = std::nullopt;

    // 已确认 engaged 后，可以安全使用 operator*。
    assert(present && *present == 42);

    // value_or 返回 int 值；它不会让 missing 从空状态变为有值。
    const int fallback = missing.value_or(-1);
    assert(fallback == -1);
    assert(!missing);

    // value() 会把未检查的空状态报告为 C++ 异常。
    try
    {
        static_cast<void>(missing.value());
    }
    catch (const std::bad_optional_access &error)
    {
        std::cout << "caught: " << error.what() << '\n';
    }
}

### 4. `optional<bool>` 有两层布尔语义

`optional<bool>` 可以表示三种业务状态：未提供、明确为 false、明确为 true。但 `if (option)` 检查的是“是否提供”，不是 contained bool 的真假。

这类类型容易被误读。需要业务值时先确认 engaged，再读取 `*option`。

In [ ]:
// 本步骤：区分“包含 false”和“完全没有 bool”。
{
    const std::optional<bool> disabled = false;
    const std::optional<bool> unspecified = std::nullopt;

    // disabled 包含值，所以 optional 自身的布尔检查为 true。
    assert(disabled.has_value());
    assert(static_cast<bool>(disabled));

    // 解引用后才得到业务值 false。
    assert(*disabled == false);

    // unspecified 才表示调用方没有提供布尔值。
    assert(!unspecified.has_value());
}

### 5. optional 管理 contained object 的生命周期

当 `optional<T>` 处于 engaged 状态时，它拥有一个 `T` 对象。`reset()`、赋值为 `nullopt` 或 optional 自身析构都会销毁这个对象。

下面用日志观察 `emplace()` 建立生命周期、`reset()` 立即结束生命周期。

In [ ]:
// 本步骤：定义带构造和析构日志的值类型，用于观察 optional 的所有权。
class TrackedValue
{
public:
    explicit TrackedValue(std::string label)
        : label_(label)
    {
        // 构造日志表示 contained object 的生命周期开始。
        std::cout << label_ << " constructed\n";
    }

    ~TrackedValue() noexcept
    {
        // 析构日志表示 optional 已结束 contained object 的生命周期。
        std::cout << label_ << " destroyed\n";
    }

    const std::string &label() const noexcept
    {
        // 返回只读借用，调用方不能让它超过 TrackedValue 生命周期。
        return label_;
    }

private:
    std::string label_;
};

In [ ]:
// 本步骤：在 optional 中构造并主动销毁对象，定位生命周期边界。
{
    std::optional<TrackedValue> value;

    // emplace 直接建立 contained object。
    value.emplace("payload");
    assert(value->label() == "payload");

    // reset 立即触发析构，而不是等待 optional 离开外层作用域。
    value.reset();
    assert(!value);

    std::cout << "optional is empty\n";
}

### 6. optional 只继承 `T` 自身的所有权语义

`optional<std::string>` 拥有 string 值；复制 optional 会复制 contained string。`optional<std::string_view>` 只“可选地保存一个借用”，不会拥有字符。

标准 `std::optional<T&>` 不成立。若确实需要可选借用，可使用 pointer、`optional<reference_wrapper<T>>` 或 view 类型，但必须把 owner 生命周期写清楚。

In [ ]:
// 本步骤：对比 optional 中的拥有值与非拥有 string_view。
{
    std::optional<std::string> owned = std::string("Kotlin");
    std::optional<std::string> copy = owned;

    // 修改副本的 contained string，不影响原 optional。
    copy->append("/Native");
    assert(*owned == "Kotlin");
    assert(*copy == "Kotlin/Native");

    std::string owner = "Native";
    std::optional<std::string_view> borrowed = owner;

    // view 读取 owner 的当前字符，不是创建时的快照。
    owner[0] = 'n';
    assert(*borrowed == "native");

    // 清空 optional 只销毁 view，不销毁它借用的 owner。
    borrowed.reset();
    assert(owner == "native");
}

下面两种写法都会留下悬空借用，只作为反例阅读：

```cpp
std::optional<std::string_view> bad_view()
{
    std::string local = "temporary";
    return local; // local 析构后，contained view 悬空
}

std::optional<const int *> bad_pointer()
{
    int local = 42;
    return &local; // optional 有值，但 pointer 已经失效
}
```

“optional 处于 engaged 状态”只说明它包含一个 `T`，不保证 `T` 内部指针或 view 仍然有效。

### 7. 缺失不是错误

optional 适合表达调用成功但没有结果：

- map 中没有对应 key；
- 配置项没有提供；
- 过滤后没有首个元素。

若还要区分无效参数、I/O 失败、解析失败和未找到，仅用空 optional 会丢失原因。此时应使用异常、明确的 result/status 类型，或在工具链支持时使用 `std::expected<T, E>`。

API 设计前先回答：调用方只需要知道“有或没有”，还是必须根据失败原因采取不同动作？

### 8. C ABI 使用状态码与 out parameter

`std::optional<T>` 是 C++ 标准库类型，不能直接出现在稳定 C ABI 中。简单接口可以用 `bool + out parameter` 表达找到/未找到；若还存在真正错误，应使用可扩展状态码：

```c
typedef enum sdk_status {
    SDK_STATUS_OK,
    SDK_STATUS_NOT_FOUND,
    SDK_STATUS_INVALID_ARGUMENT
} sdk_status;

sdk_status sdk_find_age(const char *name, int *out_age);
```

只有状态为 OK 时，调用方才能读取 out parameter。

In [ ]:
// 本步骤：用 C 状态码重写 optional 查询，并在边界验证裸指针。
enum sdk_status
{
    SDK_STATUS_OK,
    SDK_STATUS_NOT_FOUND,
    SDK_STATUS_INVALID_ARGUMENT
};

extern "C" sdk_status sdk_find_age(
    const char *name,
    int *out_age) noexcept
{
    // C ABI 不能依赖异常报告空指针，因此先显式校验参数。
    if (name == nullptr || out_age == nullptr)
    {
        return SDK_STATUS_INVALID_ARGUMENT;
    }

    const std::string_view input(name);

    // 仅在成功状态下写入 out parameter。
    if (input == "Bob")
    {
        *out_age = 20;
        return SDK_STATUS_OK;
    }

    if (input == "Alice")
    {
        *out_age = 21;
        return SDK_STATUS_OK;
    }

    return SDK_STATUS_NOT_FOUND;
}

In [ ]:
// 本步骤：分别验证成功、未找到和参数错误三条 C ABI 路径。
{
    int age = -1;

    // 成功时状态为 OK，out parameter 得到有效结果。
    const sdk_status found =
        sdk_find_age("Bob", &age);
    assert(found == SDK_STATUS_OK);
    assert(age == 20);

    // 未找到时保留原 out value；调用方不能把它当作结果读取。
    age = -1;
    const sdk_status missing =
        sdk_find_age("Unknown", &age);
    assert(missing == SDK_STATUS_NOT_FOUND);
    assert(age == -1);

    // 空 name 被识别为调用错误，而不是普通的“未找到”。
    const sdk_status invalid =
        sdk_find_age(nullptr, &age);
    assert(invalid == SDK_STATUS_INVALID_ARGUMENT);
}

### 9. Kotlin/Native wrapper 恢复惯用语义

Kotlin wrapper 可以把 C 状态协议重新映射为上层语义：

```kotlin
fun findAge(name: String): Int? =
    when (val result = nativeFindAge(name)) {
        is Ok -> result.age
        NotFound -> null
        is Failure -> throw NativeSdkException(result.status)
    }
```

Kotlin 的 `Int?` 与 `std::optional<int>` 用途相近，但二者的内存布局和 ABI 都不同，不能直接跨边界传递。C 层负责稳定协议，Kotlin wrapper 负责把 NOT_FOUND 变成 `null`，并保留真正错误的诊断信息。

### 10. 不依赖 optional 的对象布局

optional 通常需要保存 engaged 状态和一个 `T` 的存储空间，但标准不保证具体大小、padding 或二进制布局。`sizeof(optional<T>)` 也不必等于 `sizeof(T) + 1`。

因此不要：

- 用 `memcpy` 把 optional 当作稳定序列化格式；
- 把 optional 放进跨编译器或跨语言 ABI；
- 用对象全零来假设它一定表示 `nullopt`；
- 保存 contained value 的引用后再调用 `reset()` 或 `emplace()`。

In [ ]:
// 本步骤：展示 contained value 的借用只在当前 engaged 生命周期内有效。
{
    std::optional<std::string> value = "first";
    const std::string *borrowed = &value.value();

    // 在 reset 前，借用指向 optional 当前拥有的 string。
    assert(*borrowed == "first");

    // reset 会销毁 string；从这一行之后绝不能再解引用 borrowed。
    value.reset();

    // 重新 emplace 建立的是新的 contained object，必须重新取得借用。
    value.emplace("second");
    borrowed = &value.value();
    assert(*borrowed == "second");
}

### 11. 使用 optional 前的检查清单

- 缺失是否是正常、预期的业务状态？
- 调用方是否需要知道缺失或失败的具体原因？
- `T` 是拥有值、裸指针还是非拥有 view？
- 读取前是否检查 engaged，还是明确选择 `value()` 的异常策略？
- `value_or()` 返回副本是否符合成本和生命周期预期？
- contained value 的 reference/pointer 是否跨过了 `reset()`、`emplace()` 或 optional 析构？
- 跨 C ABI 时是否已转换为明确状态码和只在成功时有效的输出参数？

### 本实验结论

`std::optional<T>` 是一个拥有型状态容器：engaged 时包含并管理一个 `T`，disengaged 时不包含值。它适合在返回类型中显式表达正常缺失，避免 magic value 和不必要的裸指针协议。

optional 只改变“值是否存在”，不会改变 `T` 自己的所有权语义。`optional<string>` 拥有字符，而 `optional<string_view>` 仍然只是可能存在的借用；reset、emplace 和析构都会结束旧 contained value 的生命周期。

跨 C ABI 时应转换为 bool/status + out parameter，不能暴露 C++ optional 布局。Kotlin wrapper 可以把 NOT_FOUND 映射为 nullable 值，但真正错误仍应保留独立的状态和诊断。